# This file is for experimenting and seeing what works and how. All the final code will be available in main.py.
- Now I'm using the simplegmail library to parse mails.
- Wish me luck monekeys.

In [50]:
from simplegmail import Gmail

In [51]:
gmail = Gmail(creds_file="token.pickle", client_secret_file="credentials.json") 

In [53]:
messages = gmail.get_messages(query = "newer_than:200d")

In [54]:
len(messages)

517

In [57]:
mtypes = set()
for message in messages:
    try:
        mtypes.add(message.headers["Content-Type"].split(";")[0]) # get mtypes. 
    except KeyError:
        pass # just not going to handle this.
# this library returns content type as a string, such that:
# "content-type: multipart/...; boundary: xxx". so when we split at semicolon, and take first element of the list,
# we get content type yessir ^^

In [58]:
mtype_classified = {m: [] for m in mtypes} # dict for classifying

In [60]:
for message in messages:
    try:
        mtype_classified[message.headers["Content-Type"].split(";")[0]].append(message)
    except KeyError:
        pass

In [61]:
mtype_classified["multipart/related"] # example

[Message(to: undisclosed-recipients:;, from: Elliott P <elliott@blueprint.hackclub.com>, id: 19e2e2cbf90369b7),
 Message(to: syedmrihaan@gmail.com, from: ServiceNow Developer Program <devportalprod@service-now.com>, id: 19c7f45ff9daa406),
 Message(to: syedmrihaan@gmail.com, from: ServiceNow Developer Program <devportalprod@service-now.com>, id: 19c245241806ecb5),
 Message(to: Muhammad Rihaan <syedmrihaan@gmail.com>, from: Nishad Meharaj <nishad.meharaj@gmail.com>, id: 19bdb32db04f92e9),
 Message(to: Muhammad Rihaan <syedmrihaan@gmail.com>, from: Nishad Meharaj <nishad.meharaj@gmail.com>, id: 19b935b035166aa8),
 Message(to: undisclosed-recipients:;, from: Elliott P <elliott@blueprint.hackclub.com>, id: 19e2e2cbf90369b7),
 Message(to: syedmrihaan@gmail.com, from: ServiceNow Developer Program <devportalprod@service-now.com>, id: 19c7f45ff9daa406),
 Message(to: syedmrihaan@gmail.com, from: ServiceNow Developer Program <devportalprod@service-now.com>, id: 19c245241806ecb5),
 Message(to: Muh

In [62]:
message_count = 0

for key in mtype_classified.keys():
    for message in mtype_classified[key]:
        message_count += 1

In [63]:
message_count

996

### ***Nice, it's working.***

btw im borrowing this plan from the previous notebook.

***Data Collection***
- Collect message ID 
- Collect sender name 
- Collect sender domain, sender local part 
- Collect combined text(subject + body) 
- Collect word count of body 
- Collect number of numerical characters(digits) in body. 
- Collect link count(useful for promotional mails or maybe school mails? idk) 
- Collect currency count(useful for promotional i think) 
- Collect presence of otp-like numbers 
- Collect exclamation mark count(spammy emails) 
- Also record contains_html, and contains_plain.
- Also record presence of html structures like image_count, button_count, etc...

***Final Dataframe***
- Sender info 
- Combined text
- Word count
- Digit count
- Mail folder
- Message ID(DO NOT TRAIN ON THIS)

***Marking Targets***
- Perform unsupervised KMeans to get clusters.

***Model and Final pipeline***
- Gaussian naive bayes.

In [64]:
"""

- Collect message ID ✅
- Collect sender name ✅
- Collect sender domain, sender local part ✅
- Collect combined text(subject + body) ✅
- Collect word count of body ✅
- Collect number of numerical characters(digits) in body. ✅
- Collect link count(useful for promotional mails or maybe school mails? idk) ✅
- Collect currency count(useful for promotional i think) ✅
- Collect presence of otp-like numbers ✅
- Collect exclamation mark count(spammy emails) ✅
- Also record contains_html, and contains_plain. ✅

"""

from email.utils import parseaddr
import html, re
from bs4 import BeautifulSoup

# function to clean mails of special chars

def cleaner(text):
    cleaned = html.unescape(text)
    cleaned = re.sub(r'[\u200c\u200b\u200d\ufeff\xad]', "", cleaned) 
    # [] => match any one char, an since every char is unicode, regex treats \u200c(for eg) as a single char
    cleaned = " ".join(cleaned.split())
    return cleaned

# quick function to use expressions based on which type of msg we have

def parse_mail(msg):
    msg_id = msg.id
    sender = msg.sender
    sender_name, sender_addr = parseaddr(sender)
    sender_local, sender_domain = sender_addr.rsplit("@", 1)
    subject = cleaner(msg.subject)

    contains_html, contains_plain = int(bool(msg.html)), int(bool(msg.plain))

    soup = BeautifulSoup(msg.html, "html.parser") if contains_html else "" # for html

    body = re.sub(r"https?://\S+", "",cleaner(msg.plain)) if not contains_html else cleaner(soup.get_text(strip = True, separator = " "))# to remove links
    combined_text = f"{subject} {body}"

    word_count = len(body.split())

    digit_count = sum([char.isdigit() for char in body])

    link_count = len(soup.find_all("a", href=True)) if contains_html else re.findall(r"https?://S+", body)

    currency_count = sum([char in ["₹", "$"] for char in body])

    otp_count = len(re.findall(r"\b\d{4,8}\b", body))

    exclamation_count = sum([char == "!" for char in body])

    image_count = 0 if not contains_html else len(soup.find_all("img"))

    # final record

    record = {"id": msg_id,
              "contains_html": contains_html,
              "contains_plain_text": contains_plain,
              "sender_name": sender_name,
              "sender_local": sender_local,
              "sender_address": sender_addr,
              "comb_text": combined_text if len(combined_text.strip()) > 0 else None, # because empty records are noise creators
              "word_count": word_count,
              "digit_count": digit_count,
              "link_count": link_count,
              "currency_count": currency_count,
              "otp_count": otp_count,
              "exclamation_count": exclamation_count,
              "image_count": image_count
              }
    
    return record

In [65]:
records = [parse_mail(message) for message in messages][:]


In [66]:
import pandas as pd

df = pd.DataFrame(records).set_index("id").dropna()

In [67]:
df.head()

,contains_html,contains_plain_text,sender_name,sender_local,sender_address,comb_text,word_count,digit_count,link_count,currency_count,otp_count,exclamation_count,image_count
id,,,,,,,,,,,,,
19ec1cd1706a637f,1,1,Google,noreply-accounts,noreply-accounts@google.com,You shared some Google Account data with Slack...,295,33,3,0,3,0,12
19ec1be81dcff6cc,1,1,Hack Club,auth,auth@hackclub.com,752-267 is your Hack Club login code Your code...,73,18,1,0,0,2,1
19ec1b94b050434a,1,1,Stella Clark,stella,stella@joinspeakup.com,Invitation | Research Publication Fellowship M...,584,33,1,0,3,2,1
19ec04f3f568b7a0,1,0,RVRG FM Helpdesk,donotreply,donotreply@apnacomplex.com,[Rainbow Vistas @ Rock Garden] New Notice: UPD...,156,17,8,0,2,1,8
19ebe64e13ebe24d,1,1,Stephen from Render,hello,hello@render.com,The easiest way to launch scalable web service...,174,3,5,0,0,0,1


In [68]:
df[df["comb_text"].str.len() == 0]

,contains_html,contains_plain_text,sender_name,sender_local,sender_address,comb_text,word_count,digit_count,link_count,currency_count,otp_count,exclamation_count,image_count
id,,,,,,,,,,,,,


### **It's been a few hours.**

### Lets grind :)

# ***Plan***
- Perform TF-IDF vectorization in a pipeline probably
- Perform kmeans unsupervised.
- If i get this to work then 90% of my project is done ^^

In [69]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

In [70]:
preprocessor = ColumnTransformer([
    ("tf-idf", TfidfVectorizer(
        stop_words = "english", #this means to remove english filler wordds(like the, a, and,etc...)
        max_features = 2500,
        min_df = 5,
        max_df = 0.95,
        ngram_range = (1, 3)
        
    ), "comb_text"),
    ("cats", OneHotEncoder(drop = "first", sparse_output=False), ["sender_name", "sender_local", "sender_address"]),
    ("numerical", StandardScaler(), make_column_selector(dtype_include="int64"))
])

In [71]:
vectorized_df = preprocessor.fit_transform(df)

### **Done with TF-IDF**

# **Unsupervised Kmeans time**

In [72]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [73]:
best_score = -1
best_k = 0
best_kmeans = None

for k in range(3, 15):
    kmeans = KMeans(
        n_clusters = k,
        random_state = 11,
        n_init = 10
    )

    labels = kmeans.fit_predict(vectorized_df)
    score = silhouette_score(vectorized_df, labels) # use silhouette score to judge efficiency of k clusters

    if score > best_score:
        best_score = score
        best_k = k
        best_kmeans = kmeans

In [74]:
best_kmeans

,"n_clusters n_clusters: int, default=8The number of clusters to form as well as the number ofcentroids to generate.For an example of how to choose an optimal value for `n_clusters` refer to:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_silhouette_analysis.py`.",3
,"init init: {'k-means++', 'random'}, callable or array-like of shape (n_clusters, n_features), default='k-means++'Method for initialization:* 'k-means++' : selects initial cluster centroids using sampling based on an empirical probability distribution of the points' contribution to the overall inertia. This technique speeds up convergence. The algorithm implemented is ""greedy k-means++"". It differs from the vanilla k-means++ by making several trials at each sampling step and choosing the best centroid among them.* 'random': choose `n_clusters` observations (rows) at random from data for the initial centroids.* If an array is passed, it should be of shape (n_clusters, n_features) and gives the initial centers.* If a callable is passed, it should take arguments X, n_clusters and a random state and return an initialization.For an example of how to use the different `init` strategies, see:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_digits.py`.For an evaluation of the impact of initialization, see the example:ref:`sphx_glr_auto_examples_cluster_plot_kmeans_stability_low_dim_dense.py`.",'k-means++'
,"n_init n_init: 'auto' or int, default='auto'Number of times the k-means algorithm is run with different centroidseeds. The final results is the best output of `n_init` consecutive runsin terms of inertia. Several runs are recommended for sparsehigh-dimensional problems (see :ref:`kmeans_sparse_high_dim`).When `n_init='auto'`, the number of runs depends on the value of init:10 if using `init='random'` or `init` is a callable;1 if using `init='k-means++'` or `init` is an array-like... versionadded:: 1.2 Added 'auto' option for `n_init`... versionchanged:: 1.4 Default value for `n_init` changed to `'auto'`.",10
,"max_iter max_iter: int, default=300Maximum number of iterations of the k-means algorithm for asingle run.",300
,"tol tol: float, default=1e-4Relative tolerance with regards to Frobenius norm of the differencein the cluster centers of two consecutive iterations to declareconvergence.",0.0001
,"verbose verbose: int, default=0Verbosity mode.",0
,"random_state random_state: int, RandomState instance or None, default=NoneDetermines random number generation for centroid initialization. Usean int to make the randomness deterministic.See :term:`Glossary `.",11
,"copy_x copy_x: bool, default=TrueWhen pre-computing distances it is more numerically accurate to centerthe data first. If copy_x is True (default), then the original data isnot modified. If False, the original data is modified, and put backbefore the function returns, but small numerical differences may beintroduced by subtracting and then adding the data mean. Note that ifthe original data is not C-contiguous, a copy will be made even ifcopy_x is False. If the original data is sparse, but not in CSR format,a copy will be made even if copy_x is False.",True
,"algorithm algorithm: {""lloyd"", ""elkan""}, default=""lloyd""K-means algorithm to use. The classical EM-style algorithm is `""lloyd""`.The `""elkan""` variation can be more efficient on some datasets withwell-defined clusters, by using the triangle inequality. However it'smore memory intensive due to the allocation of an extra array of shape`(n_samples, n_clusters)`... versionchanged:: 0.18 Added Elkan algorithm.. versionchanged:: 1.1 Renamed ""full"" to ""lloyd"", and deprecated ""auto"" and ""full"". Changed ""auto"" to use ""lloyd"" instead of ""elkan"".",'lloyd'


In [75]:
best_k, best_score

(3, 0.38413246393146366)

In [76]:
clusters = best_kmeans.predict(vectorized_df)

In [77]:
df["cluster"] = clusters.astype("str") # as string so that its easier to handle in pipeline.

In [78]:
df.cluster

id
19ec1cd1706a637f    1
19ec1be81dcff6cc    1
19ec1b94b050434a    1
19ec04f3f568b7a0    1
19ebe64e13ebe24d    1
                   ..
19acd8fb52a8b03f    1
19acd8d174980999    1
19ac82a43eeea155    1
19ac590c5057cf64    1
19ac41b2e9094852    1
Name: cluster, Length: 490, dtype: str

In [79]:
[str(c) for c in sorted(clusters)][:]

['0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '0',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1',
 '1'

In [80]:
from sklearn.preprocessing import OrdinalEncoder
import numpy as np

new_preprocessor = ColumnTransformer([
    ("tf-idf", TfidfVectorizer(
        stop_words = "english", #this means to remove english filler wordds(like the, a, and,etc...)
        max_features = 2500,
        
    ), "comb_text"),
    ("cats", OneHotEncoder(drop = "first", sparse_output=False), ["sender_name", "sender_local", "sender_address"]),
    ("numerical", StandardScaler(), make_column_selector(dtype_include="int64")),
    ("cluster", OrdinalEncoder(categories=[[str(c) for c in np.unique(clusters)][:]]), ["cluster"])
])

In [81]:
clustered_df = new_preprocessor.fit_transform(df)

In [82]:
df[["comb_text", "cluster"]].head(n = 15)

,comb_text,cluster
id,,
19ec1cd1706a637f,You shared some Google Account data with Slack...,1
19ec1be81dcff6cc,752-267 is your Hack Club login code Your code...,1
19ec1b94b050434a,Invitation | Research Publication Fellowship M...,1
19ec04f3f568b7a0,[Rainbow Vistas @ Rock Garden] New Notice: UPD...,1
19ebe64e13ebe24d,The easiest way to launch scalable web service...,1
19ebd34a54c77f04,"NotARoomba sent you messages Hack Club 112,958...",0
19ebc7967c7afafb,Competition Launch: AI Agent Security: Multi-S...,1
19ebbd68603657b0,💡 Explore GitGuardian Resources 2/2 Master sec...,1
19eb92c3489aca11,New privacy settings for Search services and G...,1


In [83]:
new_clusters = dict()
for cluster in df["cluster"].unique():
    print("=" * 50)
    print(f"CLUSTER {cluster}")

    text = df.loc[
        df["cluster"] == cluster,
        "comb_text"
    ]
    vectorizer = preprocessor.named_transformers_["tf-idf"]
    vectorized_text = vectorizer.transform(text)

    mean_score = vectorized_text.mean(axis = 0)

    top_words = (
        pd.Series(
            mean_score.A1,
            index = vectorizer.get_feature_names_out()
        )
        .sort_values(ascending=False)
        .head(10)
    )

    print(top_words)

    new_clusters[cluster] = input("The program detected a lot of mails with these key words being quite frequent. These mails will be moved into a particular folder, and new incoming mails will be sorted on the basis of these keywords. What would you like to name this folder? ")

CLUSTER 1
account     0.033199
email       0.026893
app         0.026725
com         0.026465
code        0.026238
download    0.025478
session     0.024403
ai          0.022339
payment     0.021760
google      0.021349
dtype: float64
CLUSTER 0
nasa             0.302144
stem             0.082525
gov              0.081147
nasa gov         0.081147
apple            0.078991
space            0.074925
science          0.059226
resources        0.055132
learning         0.052664
opportunities    0.049391
dtype: float64
CLUSTER 2
mo          0.326307
apple       0.291806
00          0.181080
00 mo mo    0.175045
mo mo       0.170593
00 mo       0.166666
buy         0.154928
trade       0.152998
card        0.126459
cashback    0.121909
dtype: float64


In [84]:
new_clusters

{'1': 'google account stuff', '0': 'stem learning', '2': 'spam'}

In [85]:
df["cluster"] = df["cluster"].str.replace(new_clusters, None)

In [86]:
df.head(10)

,contains_html,contains_plain_text,sender_name,sender_local,sender_address,comb_text,word_count,digit_count,link_count,currency_count,otp_count,exclamation_count,image_count,cluster
id,,,,,,,,,,,,,,
19ec1cd1706a637f,1,1,Google,noreply-accounts,noreply-accounts@google.com,You shared some Google Account data with Slack...,295,33,3,0,3,0,12,google account stuff
19ec1be81dcff6cc,1,1,Hack Club,auth,auth@hackclub.com,752-267 is your Hack Club login code Your code...,73,18,1,0,0,2,1,google account stuff
19ec1b94b050434a,1,1,Stella Clark,stella,stella@joinspeakup.com,Invitation | Research Publication Fellowship M...,584,33,1,0,3,2,1,google account stuff
19ec04f3f568b7a0,1,0,RVRG FM Helpdesk,donotreply,donotreply@apnacomplex.com,[Rainbow Vistas @ Rock Garden] New Notice: UPD...,156,17,8,0,2,1,8,google account stuff
19ebe64e13ebe24d,1,1,Stephen from Render,hello,hello@render.com,The easiest way to launch scalable web service...,174,3,5,0,0,0,1,google account stuff
19ebd34a54c77f04,1,1,Hack Club (via Slack),notification,notification@slack.com,"NotARoomba sent you messages Hack Club 112,958...",480,75,22,0,2,20,40,stem learning
19ebc7967c7afafb,1,1,Kaggle,no-reply,no-reply@kaggle.com,Competition Launch: AI Agent Security: Multi-S...,213,20,3,1,3,0,2,google account stuff
19ebbd68603657b0,1,1,Dwayne McDaniel,dwayne.mcdaniel,dwayne.mcdaniel@gitguardian.com,💡 Explore GitGuardian Resources 2/2 Master sec...,291,0,7,0,0,0,1,google account stuff
19eb92c3489aca11,1,1,Google,google-noreply,google-noreply@google.com,New privacy settings for Search services and G...,502,13,5,0,3,0,1,google account stuff
